# Dashboard tables freshness check

Checks table availability, max business date, target date readiness, and key join coverage.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None

In [ ]:
# Parameters
target_date = '2026-02-28'
mem_limit = '8g'
use_metadata_refresh = True

tables_cfg = [
    {'table': 'ods_alpha.scd1_agreements', 'date_col': None, 'critical': 1},
    {'table': 'ods_alpha.scd1_companies', 'date_col': None, 'critical': 1},
    {'table': 'ods_alpha.scd1_merchants', 'date_col': None, 'critical': 1},
    {'table': 'ods_alpha.scd1_pos_terminals', 'date_col': None, 'critical': 1},
    {'table': 'ods_alpha.scd1_trx', 'date_col': 'd_trx_orig', 'critical': 1},
    {'table': 'ods_alpha.scd1_trx_acq', 'date_col': None, 'critical': 1},
    {'table': 'ods_alpha.scd1_trx_int', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_r2_ip_merchants', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_r2_tariff_tune', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_r2_tariff_fix', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_r2_tariff_plan', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_depart', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_branch', 'date_col': None, 'critical': 0},
    {'table': 'ods.scd1_z_cl_corp', 'date_col': None, 'critical': 0},
    {'table': 'sandbox_ai.shestopalov_terminal_amortization_oneoff', 'date_col': None, 'critical': 0},
    {'table': 'sandbox_ai.shestopalov_feb_base_keys', 'date_col': None, 'critical': 0},
    {'table': 'sandbox_ai.shestopalov_feb_trx_small', 'date_col': None, 'critical': 0},
    {'table': 'sandbox_ai.shestopalov_feb_trx_agg_by_agr', 'date_col': None, 'critical': 0}
]

print('target_date =', target_date)
print('mem_limit =', mem_limit)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    if use_metadata_refresh:
        for cfg in tables_cfg:
            try:
                imp.execute(f"invalidate metadata {cfg['table']}")
            except Exception:
                pass

print('Impala connected')

In [ ]:
rows = []
for cfg in tables_cfg:
    table = cfg['table']
    date_col = cfg['date_col']
    critical = cfg['critical']
    try:
        sql_cnt = f"select count(*) as row_cnt from {table}"
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            cnt_df = imp.fetch(sql_cnt)
        row_cnt = int(cnt_df.loc[0, 'row_cnt']) if len(cnt_df) else 0

        max_business_date = None
        target_rows = None
        if date_col:
            sql_dt = f"""
            select
              max(to_date(cast({date_col} as timestamp))) as max_business_date,
              sum(case when to_date(cast({date_col} as timestamp)) = cast('{target_date}' as date) then 1 else 0 end) as target_rows
            from {table}
            """
            with imp:
                imp.execute(f"set MEM_LIMIT={mem_limit}")
                dt_df = imp.fetch(sql_dt)
            if len(dt_df):
                max_business_date = dt_df.loc[0, 'max_business_date']
                target_rows = int(dt_df.loc[0, 'target_rows']) if pd.notna(dt_df.loc[0, 'target_rows']) else 0

        if date_col is None:
            status = 'no_date_column'
        else:
            status = 'ready' if target_rows and target_rows > 0 else 'lagging'

        rows.append({
            'table': table,
            'critical': critical,
            'row_cnt': row_cnt,
            'date_col': date_col,
            'max_business_date': max_business_date,
            'target_rows': target_rows,
            'status': status,
            'error': None
        })
    except Exception as e:
        rows.append({
            'table': table,
            'critical': critical,
            'row_cnt': None,
            'date_col': date_col,
            'max_business_date': None,
            'target_rows': None,
            'status': 'error',
            'error': str(e)[:500]
        })

freshness_summary = pd.DataFrame(rows)
display(freshness_summary.sort_values(['critical', 'status', 'table'], ascending=[False, True, True]))
print('critical_not_ready =', len(freshness_summary[(freshness_summary['critical'] == 1) & (freshness_summary['status'].isin(['lagging', 'error']))]))

In [ ]:
# trx -> acq/int coverage on target_date
sql_cov = f"""
with trx_day as (
  select cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  where to_date(cast(t.d_trx_orig as timestamp)) = cast('{target_date}' as date)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
),
acq as (
  select distinct cast(n_trx as string) as n_trx
  from ods_alpha.scd1_trx_acq
),
ints as (
  select distinct cast(n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int
)
select
  count(*) as trx_day_cnt,
  sum(case when a.n_trx is not null then 1 else 0 end) as trx_with_acq,
  sum(case when i.n_trx is not null then 1 else 0 end) as trx_with_int,
  round(100.0 * sum(case when a.n_trx is not null then 1 else 0 end) / nullif(count(*),0), 2) as acq_coverage_pct,
  round(100.0 * sum(case when i.n_trx is not null then 1 else 0 end) / nullif(count(*),0), 2) as int_coverage_pct
from trx_day t
left join acq a on a.n_trx = t.n_trx
left join ints i on i.n_trx = t.n_trx
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    coverage_df = imp.fetch(sql_cov)

display(coverage_df)

In [ ]:
# Simple business readiness verdict
critical_bad = freshness_summary[(freshness_summary['critical'] == 1) & (freshness_summary['status'].isin(['lagging', 'error']))]
if len(critical_bad) == 0:
    print('READY: all critical sources look ready for target_date')
else:
    print('NOT READY: critical sources have issues')
    display(critical_bad[['table', 'status', 'max_business_date', 'target_rows', 'error']])